In [0]:
catalog_list=["amit"]
embedding_model_list=["databricks-qwen3-embedding-0-6b"]
schema_list=["bertopic","default"]
input_table_list=["bertopic_input"]
instruct_model_list=["databricks-meta-llama-3-3-70b-instruct"]

def add_column(table_name, column_name, column_type):
    try:
        spark.sql(f"DESCRIBE {table_name}").filter("col_name = 'text_embedding'").count()
        print(f"Column {column_name} already exists")
    except:
        spark.sql(f"""
        ALTER TABLE {table_name}
        ADD COLUMNS ({column_name} {column_type})
        """)
        print(f"Column {column_name} added")

def get_ai_merge_sql(table_name, model_name, ai_result_column, input_column ):
  sql_str=f"""  MERGE INTO {table_name} t
    USING (
      SELECT
        id,
        ai_query("{model_name}", {input_column}) AS embedding
      FROM {table_name}
      WHERE {ai_result_column} IS NULL
        AND {input_column} IS NOT NULL and {input_column}!= ''
      LIMIT 10
    ) s
    ON t.id = s.id
    WHEN MATCHED AND t.{ai_result_column} IS NULL THEN
      UPDATE SET t.{ai_result_column} = s.embedding;
  """
  return sql_str
def add_column_batch(table_name, model_name,ai_result_column,input_column, column_type, is_ai ):
    add_column(table_name, column_name=ai_result_column, column_type=column_type)
    sql_str=None
    if is_ai==True:
        sql_str=get_ai_merge_sql(table_name=table_name, model_name=model_name, ai_result_column=ai_result_column, input_column=input_column)
    while True:
        null_count = spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {table_name}
            WHERE {ai_result_column} IS NULL
            AND {input_column} IS NOT NULL and {input_column} <>""
        """).collect()[0]["c"]

        if null_count == 0:
            print(f"Done. No more NULL {ai_result_column}.")
            break

        print(f"Remaining records to embed: {null_count}. Processing next batch...")
        spark.sql(sql_str)    

